# Pre-final report: clustering product attributes

**Status:** preliminary; results below are recomputed from the checked-in similarity matrix when the code cells are run. This notebook is the single report/figure file requested; it supplements rather than replaces the method notebooks.

## Summary

We group product-feature titles that describe the same property, helping catalogue users find duplicates and normalize inconsistent wording. The current pipeline combines description TF-IDF, contextual Qwen embeddings, and RapidFuzz title matching into one pairwise similarity matrix, then compares agglomerative, density-based, and graph clustering. No ground-truth equivalence labels are present, so internal scores and manual cluster inspection are provisional evidence, not proof of business correctness.

## Problem and data

The input is `data/catalog.json`, described in the preprocessing notebook as a Salsify catalog export. The repository does not record the export URL, date, licence, or collection procedure; confirm these before final submission. Preprocessing produces 208 products, 445 unique English feature titles, and 22 descriptions that could not be paired to titles. There are 433 titles with descriptions; 202 titles occur on only one product. This is a small, sparse, unlabeled dataset with many near-synonyms and some deceptively similar but distinct feature concepts.

## Similarity construction

For every title pair, the current blend is `0.50 × semantic + 0.25 × TF-IDF + 0.25 × fuzzy`, then a `0.30` penalty is subtracted for 72 detected conflicting-concept pairs (clipped to `[0, 1]`). All clustering uses distance `D = 1 - similarity`.

- **TF-IDF / IDF:** rare description terms count more than common terms; a sparse lexical baseline that can distinguish specific properties.
- **Qwen/Qwen3-Embedding-0.6B:** contextual title/description vectors help match paraphrases and synonyms. Title and aligned-description vectors are blended 50:50; the first model run downloads about 1.2 GB.
- **RapidFuzz:** fast, normalized string matching catches spelling, punctuation, and word-order variants that embeddings or TF-IDF may miss.

The model and blend weights are documented in Part 3 of `final_report.ipynb`; only the final 445 × 445 matrix and titles are saved in `data/similarity.npz`. The component matrices are not saved, so historic weight variants cannot be independently reproduced from that file.

## Clustering methods

All four methods use the same pairwise matrix: agglomerative, DBSCAN, and HDBSCAN operate on distance `D = 1 - similarity`; spectral clustering uses similarity to build a graph. Their different mathematical assumptions lead to different kinds of groups.

### Agglomerative clustering

Begin with each title in its own cluster. Under average linkage, the distance between clusters `A` and `B` is the mean of every cross-cluster distance: `d(A,B) = sum(D[i,j] for i in A, j in B) / (|A| × |B|)`. Repeatedly merge the pair with the smallest `d(A,B)` and update the distances to the merged cluster. Stop when the next merge reaches `distance_threshold=0.70`. This builds a hierarchy but assigns every title; early merges cannot be undone.

### DBSCAN

For title `i`, define its `eps`-neighborhood as `N_eps(i) = {j : D[i,j] <= eps}`. A title is a **core point** if its neighborhood has at least `min_samples` titles (including itself). DBSCAN expands clusters through chains of neighboring core points, attaches reachable border points, and marks points not reachable from a core point as **noise**. This report tests `eps=0.20..0.68` by `0.02` and `min_samples ∈ {2,3,4,5,6,8,10,12,14}`; clusters of size one are also marked as noise. A single `eps` radius may not fit groups with different densities.

### HDBSCAN

HDBSCAN estimates local density from each point's `min_samples`-neighbor distance, then uses **mutual reachability distance** `d_mr(i,j) = max(core_distance(i), core_distance(j), D[i,j])` to reduce the influence of sparse connections. A minimum spanning tree of these distances forms a hierarchy as the density scale changes. The hierarchy is condensed using `min_cluster_size`, and the default excess-of-mass rule selects clusters that persist across scales; other titles are noise. This report tests `min_cluster_size ∈ {2,3,4,5,8,12,20}` and `min_samples ∈ {2,3,4,5,6,8}`, skipping pairs where `min_samples > min_cluster_size`.

### Spectral clustering

Build a weighted graph `W` by connecting each title to its 15 most similar neighbors, then symmetrize the graph. The normalized graph Laplacian is `L = I - Δ^(-1/2) W Δ^(-1/2)`, where `Δ` is the diagonal matrix of weighted node degrees. The eigenvectors for the `k` smallest eigenvalues embed titles in a lower-dimensional space; row-normalize this embedding and run K-means on it. Here the largest eigengap in `k=20..80` selects `k=25`; K-means uses `n_init=10` and seed 0. Unlike the distance-radius methods, this method groups by graph connectivity.

### Evaluation and current interpretation

Archived notebooks contain inconsistent/stale outputs (for example, agglomerative reports both 225 and 138 clusters). The saved spectral output reports silhouette 0.126 and a medoid DB-like score of 1.648; the table below recomputes both metrics for every method using the checked-in matrix. Earlier DBSCAN/HDBSCAN outputs used different noise-scoring rules, so compare only the recomputed table.

In the rerun, agglomerative has the highest all-title silhouette (0.1841); DBSCAN has the lowest custom medoid DB-like score (0.4917). The DB-like result can be made artificially favorable by its many noise titles, which are scored as singleton clusters with zero within-cluster scatter. The metrics therefore do not identify one winner: agglomerative is strongest by silhouette, while HDBSCAN is a more balanced density-based option.

### Weight experiments: what is known and what to test

The DBSCAN notebook contains historical notes for base weights `(semantic, TF-IDF, fuzzy)=(0.50,0.25,0.25)`, `(0.70,0.15,0.15)`, and `(0.25,0.50,0.25)`, plus a title/description embedding variant described as 25% title / 75% descriptions. Those notes report differing DBSCAN settings/scores, but the component matrices, full run table, and exact controlled procedure are absent. Treat these as **exploratory attempts, not reproducible evidence**. The baseline `W_TITLE=0.50` is in the similarity notebook.

Next: save the semantic, TF-IDF, and fuzzy matrices; test one-factor-at-a-time blends `(0.70,0.15,0.15)`, `(0.25,0.50,0.25)`, `(0.25,0.25,0.50)` and the baseline, keeping the conflict penalty and clustering grids fixed. Separately test title/description blend `W_TITLE ∈ {0.25,0.50,0.75}`. Compare pairwise ranking against a small manually labeled similar/dissimilar set, then rerun the four clustering evaluations. This separates similarity quality from clustering-parameter effects. Co-occurrence on the same product can be a weak negative/positive signal, but is not ground truth for synonymy.

Before final submission, document the catalog export provenance/licence, curate human-labeled pairs and cluster coherence checks, reconcile stale outputs, and add member names/contributions. Avoid claiming a winning algorithm from internal metrics alone.

## Reproducible results and figures

Run the code cells top to bottom from the repository root. They use the checked-in matrix and scikit-learn's HDBSCAN; they do not download or rebuild the embedding model. Figures share one t-SNE projection of the same precomputed distances; that projection is for visualization only, not a clustering metric.


In [1]:
from collections import Counter
from pathlib import Path
import time

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import AgglomerativeClustering, DBSCAN, HDBSCAN, KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

matrix_path = Path("data/similarity.npz")
if not matrix_path.exists():
    raise FileNotFoundError("Run this notebook from the repository root; data/similarity.npz was not found.")

with np.load(matrix_path) as saved:
    S = saved["similarity"].astype(float)
    attributes = saved["titles"].tolist()

n = len(attributes)
assert S.shape == (n, n), f"Expected ({n}, {n}), got {S.shape}"
assert np.allclose(S, S.T), "Similarity matrix must be symmetric"
assert np.isfinite(S).all() and S.min() >= 0 and S.max() <= 1 + 1e-9
D = np.clip(1 - S, 0, 1)
np.fill_diagonal(D, 0)
print(f"Loaded {n} titles; similarity range [{S.min():.3f}, {S.max():.3f}]; distance shape {D.shape}")

Loaded 445 titles; similarity range [0.000, 1.000]; distance shape (445, 445)


## Evaluation approach

Both reported measures are **internal metrics**: they describe the shape of the clusters under the chosen distance matrix, not whether two titles are truly equivalent.

### Silhouette (higher is better)

For title `i`, let `a(i)` be its mean distance to the other titles in its cluster, and let `b(i)` be its smallest mean distance to another cluster. Its silhouette is `s(i) = (b(i) - a(i)) / max(a(i), b(i))`; the report averages this over titles. Scores range from -1 to 1: a value near 1 means a title is closer to its own cluster than to alternatives, near 0 means clusters overlap, and a negative value suggests it may fit another cluster better. As a rough, non-universal guide, values above 0.7 often indicate strong separation, 0.5–0.7 reasonable separation, 0.25–0.5 weak separation, and below 0.25 little separation.

### Medoid Davies–Bouldin-like score (lower is better)

For each cluster, the code chooses a **medoid**—the member with the smallest total distance to the other members—and measures its scatter as the mean member-to-medoid distance. For a pair of clusters `i,j`, it computes `(scatter_i + scatter_j) / distance(medoid_i, medoid_j)`. It takes the largest such ratio for each cluster, then averages those values. Lower scores mean clusters are compact relative to the distance between their medoids. This is a custom adaptation for a precomputed distance matrix, not the standard feature-space Davies–Bouldin index; zero is ideal in the formula, but there is no universal cutoff for a useful score.

### Noise and coverage

To score all 445 titles consistently, each DBSCAN/HDBSCAN noise title is treated as a separate singleton cluster. A singleton contributes zero silhouette; its medoid scatter is also zero, which can make the custom Davies–Bouldin-like score look better. **Coverage** is the percentage assigned to a non-noise cluster, and must be read with the scores: a method may score well while leaving many titles out. The results table reports the actual coverage for each method. None of these measures establishes semantic correctness; that would require human-checked labels or cluster review.


In [2]:
def score_labels(labels, distances):
    labels = np.asarray(labels).copy()
    noise = labels == -1
    n_noise = int(noise.sum())
    n_clusters = len(set(labels[~noise].tolist()))
    full_labels = labels.copy()
    next_label = int(labels[~noise].max() + 1) if np.any(~noise) else 0
    full_labels[noise] = np.arange(next_label, next_label + n_noise)
    n_full_clusters = len(np.unique(full_labels))
    full_silhouette = float("nan")
    if 2 <= n_full_clusters < len(labels):
        full_silhouette = float(silhouette_score(distances, full_labels, metric="precomputed"))
    return {
        "labels": labels,
        "full_labels": full_labels,
        "clusters": n_clusters,
        "noise": n_noise,
        "coverage": float((~noise).mean()),
        "silhouette_all": full_silhouette,
    }


def mark_small_clusters_as_noise(labels, minimum_size=2):
    labels = np.asarray(labels).copy()
    for label in np.unique(labels):
        if label != -1 and np.count_nonzero(labels == label) < minimum_size:
            labels[labels == label] = -1
    return labels


def fair_rank(row):
    return (-row["metrics"]["silhouette_all"], -row["metrics"]["coverage"], row["metrics"]["clusters"])


# One fixed average-linkage threshold is tested here; this is not a threshold sweep.
start = time.perf_counter()
agglo_labels = AgglomerativeClustering(
    n_clusters=None, metric="precomputed", linkage="average", distance_threshold=0.70
).fit_predict(D)
agglo_metrics = score_labels(agglo_labels, D)
agglo_seconds = time.perf_counter() - start

# DBSCAN grid: eps 0.20-0.68 by 0.02, nine min_samples values; singleton clusters are relabeled as noise.
DBSCAN_EPS = np.round(np.arange(0.20, 0.70, 0.02), 2).tolist()
DBSCAN_MIN_SAMPLES = [2, 3, 4, 5, 6, 8, 10, 12, 14]
dbscan_rows = []
start = time.perf_counter()
for eps in DBSCAN_EPS:
    for min_samples in DBSCAN_MIN_SAMPLES:
        labels = DBSCAN(eps=eps, min_samples=min_samples, metric="precomputed").fit_predict(D)
        labels = mark_small_clusters_as_noise(labels, minimum_size=2)
        metrics = score_labels(labels, D)
        if metrics["clusters"] >= 2 and np.isfinite(metrics["silhouette_all"]):
            dbscan_rows.append({"eps": eps, "min_samples": min_samples, "metrics": metrics})
dbscan_seconds = time.perf_counter() - start
if not dbscan_rows:
    raise ValueError("DBSCAN grid produced no valid multi-cluster result")
dbscan_fair_best = sorted(dbscan_rows, key=fair_rank)[0]
# HDBSCAN grid: test each listed pair only when min_samples <= min_cluster_size.
HDBSCAN_MIN_CLUSTER_SIZE = [2, 3, 4, 5, 8, 12, 20]
HDBSCAN_MIN_SAMPLES = [2, 3, 4, 5, 6, 8]
hdbscan_rows = []
start = time.perf_counter()
hdbscan_attempted = 0
for min_cluster_size in HDBSCAN_MIN_CLUSTER_SIZE:
    for min_samples in HDBSCAN_MIN_SAMPLES:
        if min_samples > min_cluster_size:
            continue
        hdbscan_attempted += 1
        labels = HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric="precomputed",
            copy=True,
        ).fit_predict(D)
        metrics = score_labels(labels, D)
        if metrics["clusters"] >= 2 and np.isfinite(metrics["silhouette_all"]):
            hdbscan_rows.append({
                "min_cluster_size": min_cluster_size,
                "min_samples": min_samples,
                "metrics": metrics,
            })
hdbscan_seconds = time.perf_counter() - start
if not hdbscan_rows:
    raise ValueError("HDBSCAN grid produced no valid multi-cluster result")
hdbscan_best = sorted(hdbscan_rows, key=fair_rank)[0]

# Spectral: keep 15 nearest neighbors; choose k from eigengaps in 20-80 (the separate notebook also swept k=10-100 by 5).
# Spectral clustering, following 5_spectral.ipynb.
K_NEIGHBORS = 15
W = S.copy()
np.fill_diagonal(W, 0)
thresholds = np.sort(W, axis=1)[:, -K_NEIGHBORS][:, None]
W = np.where((W >= thresholds) | (W >= thresholds.T), W, 0)
degree = W.sum(axis=1)
inverse_sqrt_degree = 1 / np.sqrt(np.maximum(degree, 1e-12))
laplacian = np.eye(n) - W * np.outer(inverse_sqrt_degree, inverse_sqrt_degree)
eigenvalues, eigenvectors = np.linalg.eigh(laplacian)
K_MIN, K_MAX = 20, 80
gaps = np.diff(eigenvalues[: K_MAX + 1])
k_spectral = int(np.argmax(gaps[K_MIN - 1 : K_MAX]) + K_MIN)
U = eigenvectors[:, :k_spectral]
U = U / np.maximum(np.linalg.norm(U, axis=1, keepdims=True), 1e-12)
spectral_labels = KMeans(n_clusters=k_spectral, n_init=10, random_state=0).fit_predict(U)
spectral_metrics = score_labels(spectral_labels, D)
spectral_edges = int(np.count_nonzero(np.triu(W)))

clusterings = {
    "Agglomerative": {"labels": agglo_metrics["labels"], "metrics": agglo_metrics, "params": "average, threshold=0.70", "seconds": agglo_seconds},
    "DBSCAN": {"labels": dbscan_fair_best["metrics"]["labels"], "metrics": dbscan_fair_best["metrics"], "params": f"eps={dbscan_fair_best['eps']:.2f}, min_samples={dbscan_fair_best['min_samples']} (common-score selection)", "seconds": dbscan_seconds},
    "HDBSCAN": {"labels": hdbscan_best["metrics"]["labels"], "metrics": hdbscan_best["metrics"], "params": f"min_cluster_size={hdbscan_best['min_cluster_size']}, min_samples={hdbscan_best['min_samples']}", "seconds": hdbscan_seconds},
    "Spectral": {"labels": spectral_metrics["labels"], "metrics": spectral_metrics, "params": f"k={k_spectral}, k_neighbors={K_NEIGHBORS}, seed=0", "seconds": float("nan")},
}

print(f"DBSCAN grid: {len(DBSCAN_EPS)} eps × {len(DBSCAN_MIN_SAMPLES)} min_samples = {len(DBSCAN_EPS) * len(DBSCAN_MIN_SAMPLES)} attempted; {len(dbscan_rows)} valid multi-cluster runs")
print(f"HDBSCAN grid: {hdbscan_attempted} parameter pairs tested; {len(hdbscan_rows)} produced valid multi-cluster scores (min_samples > min_cluster_size skipped)")
print(f"Spectral graph: {spectral_edges} undirected edges; eigengap k={k_spectral}")

DBSCAN grid: 25 eps × 9 min_samples = 225 attempted; 129 valid multi-cluster runs
HDBSCAN grid: 28 parameter pairs tested; 26 produced valid multi-cluster scores (min_samples > min_cluster_size skipped)
Spectral graph: 4618 undirected edges; eigengap k=25


In [3]:
def medoid_db_like_precomputed(labels, distances):
    cluster_ids = np.unique(labels)
    if len(cluster_ids) < 2:
        return float("nan")

    medoids = []
    scatters = []
    for cluster_id in cluster_ids:
        members = np.flatnonzero(labels == cluster_id)
        within = distances[np.ix_(members, members)]
        medoid = members[np.argmin(within.sum(axis=1))]
        medoids.append(medoid)
        scatters.append(float(distances[members, medoid].mean()))

    separation = distances[np.ix_(medoids, medoids)].copy()
    np.fill_diagonal(separation, np.inf)
    ratios = (np.asarray(scatters)[:, None] + np.asarray(scatters)[None, :]) / np.maximum(separation, 1e-12)
    return float(ratios.max(axis=1).mean())


print(f"{'Method':<16} {'Parameters':<48} {'Clusters':>8} {'Coverage (%)':>15} {'Silhouette':>11} {'Medoid DB-like':>16}")
print('-' * 119)
for name, result in clusterings.items():
    metrics = result["metrics"]
    db_index = medoid_db_like_precomputed(metrics["full_labels"], D)
    print(f"{name:<16} {result['params']:<48} {metrics['clusters']:>8} {metrics['coverage']:>14.1%} {metrics['silhouette_all']:>11.4f} {db_index:>16.4f}")


Method           Parameters                                       Clusters    Coverage (%)  Silhouette   Medoid DB-like
-----------------------------------------------------------------------------------------------------------------------
Agglomerative    average, threshold=0.70                               138         100.0%      0.1841           0.9114
DBSCAN           eps=0.40, min_samples=2 (common-score selection)       59          37.1%      0.1380           0.4917
HDBSCAN          min_cluster_size=2, min_samples=2                     106          73.3%      0.1778           0.7245
Spectral         k=25, k_neighbors=15, seed=0                           25         100.0%      0.1217           1.6526


### Results graphs

Each panel shows the same t-SNE projection with labels from one method. A singleton/noise-aware all-title silhouette is shown in the title. t-SNE can distort distances; use it to inspect broad grouping only.

In [4]:
embedding = TSNE(
    n_components=2,
    metric="precomputed",
    init="random",
    learning_rate="auto",
    perplexity=30,
    random_state=0,
).fit_transform(D)

figure = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        f"{name}: {result['metrics']['clusters']} clusters; silhouette={result['metrics']['silhouette_all']:.3f}"
        for name, result in clusterings.items()
    ],
)
for index, (name, result) in enumerate(clusterings.items()):
    row, col = index // 2 + 1, index % 2 + 1
    labels = result["labels"]
    noise = labels == -1
    assigned = ~noise
    figure.add_trace(
        go.Scattergl(
            x=embedding[assigned, 0],
            y=embedding[assigned, 1],
            mode="markers",
            marker={"color": labels[assigned], "colorscale": "Turbo", "size": 7, "opacity": 0.8, "showscale": False},
            text=np.asarray(attributes)[assigned],
            hovertemplate="%{text}<extra>cluster</extra>",
            showlegend=False,
        ),
        row=row,
        col=col,
    )
    if noise.any():
        figure.add_trace(
            go.Scattergl(
                x=embedding[noise, 0],
                y=embedding[noise, 1],
                mode="markers",
                marker={"color": "#999999", "size": 6, "symbol": "x", "opacity": 0.65},
                text=np.asarray(attributes)[noise],
                hovertemplate="%{text}<extra>noise</extra>",
                showlegend=False,
            ),
            row=row,
            col=col,
        )
figure.update_xaxes(title_text="t-SNE 1")
figure.update_yaxes(title_text="t-SNE 2")
figure.update_layout(template="simple_white", height=850, width=1100, title="Product-attribute clusters (shared distance projection)")
figure.show()

### Density parameter sensitivity

Heatmaps use the same all-title silhouette rule. Blank cells are invalid/nonexistent combinations. Scores are internal metrics; do not select a configuration without checking coverage and semantic coherence.

In [5]:
dbscan_grid = np.full((len(DBSCAN_MIN_SAMPLES), len(DBSCAN_EPS)), np.nan)
dbscan_eps_index = {value: index for index, value in enumerate(DBSCAN_EPS)}
dbscan_ms_index = {value: index for index, value in enumerate(DBSCAN_MIN_SAMPLES)}
for row in dbscan_rows:
    dbscan_grid[dbscan_ms_index[row['min_samples']], dbscan_eps_index[row['eps']]] = row['metrics']['silhouette_all']

hdbscan_grid = np.full((len(HDBSCAN_MIN_SAMPLES), len(HDBSCAN_MIN_CLUSTER_SIZE)), np.nan)
hdbscan_mcs_index = {value: index for index, value in enumerate(HDBSCAN_MIN_CLUSTER_SIZE)}
hdbscan_ms_index = {value: index for index, value in enumerate(HDBSCAN_MIN_SAMPLES)}
for row in hdbscan_rows:
    hdbscan_grid[hdbscan_ms_index[row['min_samples']], hdbscan_mcs_index[row['min_cluster_size']]] = row['metrics']['silhouette_all']

grid_figure = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("DBSCAN: eps × min_samples", "HDBSCAN: min_cluster_size × min_samples"),
)
grid_figure.add_trace(
    go.Heatmap(z=dbscan_grid, x=DBSCAN_EPS, y=DBSCAN_MIN_SAMPLES, colorscale="Viridis", colorbar={"title": "Silhouette"}, hovertemplate="eps=%{x}, min_samples=%{y}<br>score=%{z:.3f}<extra></extra>"),
    row=1,
    col=1,
)
grid_figure.add_trace(
    go.Heatmap(z=hdbscan_grid, x=HDBSCAN_MIN_CLUSTER_SIZE, y=HDBSCAN_MIN_SAMPLES, colorscale="Viridis", showscale=False, hovertemplate="min_cluster_size=%{x}, min_samples=%{y}<br>score=%{z:.3f}<extra></extra>"),
    row=1,
    col=2,
)
grid_figure.update_xaxes(title_text="eps", row=1, col=1)
grid_figure.update_xaxes(title_text="min_cluster_size", row=1, col=2)
grid_figure.update_yaxes(title_text="min_samples", row=1, col=1)
grid_figure.update_yaxes(title_text="min_samples", row=1, col=2)
grid_figure.update_layout(template="simple_white", height=480, width=1100, title="Density-clustering grid comparison")
grid_figure.show()

## Tested parameter conditions

The table separates parameter searches from fixed settings. For DBSCAN and HDBSCAN, each noise title is treated as its own singleton when computing the all-title silhouette; coverage separately shows what fraction was assigned to actual clusters. A grid setting is valid when it produces at least two non-noise clusters and a defined score.

| Method | Conditions tested | Coverage | Result of the current search |
|---|---|---:|---|
| Agglomerative | Average linkage; precomputed distance; fixed `distance_threshold=0.70`; no threshold sweep in this notebook. | 100% | 138 clusters; all-title silhouette 0.1841; medoid DB-like score 0.9114. |
| DBSCAN | 225 combinations: `eps=0.20..0.68` by 0.02 and `min_samples ∈ {2,3,4,5,6,8,10,12,14}`; clusters smaller than 2 marked as noise. 129 combinations gave valid scores. | 37.1% (165/445) | `eps=0.40`, `min_samples=2`; all-title silhouette 0.1380; medoid DB-like score 0.4917. |
| HDBSCAN | `min_cluster_size ∈ {2,3,4,5,8,12,20}` and `min_samples ∈ {2,3,4,5,6,8}`; 28 pairs tested after skipping `min_samples > min_cluster_size`; 26 gave valid scores. | 73.3% (326/445) | `min_cluster_size=2`, `min_samples=2`; all-title silhouette 0.1778; medoid DB-like score 0.7245. Both selected values are at the low edge of the tested grid. |
| Spectral | Fixed `k_neighbors=15`; eigengap searched for `k=20..80`, selecting `k=25` (seed 0, KMeans `n_init=10`). The original spectral notebook also evaluated `k=10..100` by 5. | 100% | All-title silhouette 0.1217; medoid DB-like score 1.6526. |
| Similarity blend | Saved baseline: semantic 0.50, TF-IDF 0.25, fuzzy 0.25; title/description embedding blend 0.50/0.50. Alternative weights appear in old notes but cannot be verified because component matrices and full runs were not saved. | — | Baseline is reproducible from the saved final matrix; alternatives are exploratory notes, not confirmed controlled tests. |

## What the results suggest for this dataset

- **Agglomerative** assigns every title, but its silhouette indicates little separation at the selected threshold.
- **DBSCAN** leaves 280 titles as noise at its selected setting. Its relatively low medoid DB-like score should be read cautiously because singleton noise clusters have zero scatter in that formula.
- **HDBSCAN** assigns more titles than DBSCAN and has a higher all-title silhouette, but its selected values are at the low edge of the tested parameter grid.
- **Spectral** assigns every title and offers a graph-based alternative; its silhouette is lowest in this run.

**Takeaway:** no method is established as semantically correct. The table shows trade-offs in separation, coverage, and the custom medoid score; use the plots and inspect cluster members before treating any grouping as interchangeable.

## Useful literature

These research articles were published within the last ten years. They provide background for the methods and signals used here; none validates this specific catalog.

- [HDBSCAN: Hierarchical density based clustering (2017)](https://doi.org/10.21105/joss.00205) introduces the HDBSCAN implementation and is directly relevant to the density hierarchy and cluster-stability explanation.
- [Comparison of Internal Clustering Validation Indices for Prototype-Based Clustering (2017)](https://doi.org/10.3390/a10030105) compares internal validation indices. It helps contextualize silhouette and Davies–Bouldin-style scores, though its experiments are prototype-based.
- [Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks (2019)](https://aclanthology.org/D19-1410/) explains sentence embeddings for semantic similarity, relevant to matching feature-title paraphrases.
- [Deep Entity Matching with Pre-Trained Language Models (2020; VLDB 2021)](https://arxiv.org/abs/2004.00584) studies matching records with pretrained language models and is relevant to duplicate-feature matching.
- [BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models (2021)](https://arxiv.org/abs/2104.08663) compares lexical and neural methods across varied datasets, providing context for combining lexical and semantic signals.
- [MTEB: Massive Text Embedding Benchmark (2023)](https://aclanthology.org/2023.eacl-main.148/) evaluates embeddings across tasks, including clustering, and finds no model is best for every task; this supports validating embeddings on the target data.
- [Qwen3 Embedding: Advancing Text Embedding and Reranking Through Foundation Models (2025)](https://arxiv.org/abs/2506.05176) describes the Qwen3 embedding family used here. Its benchmarks explain the model but do not establish performance on this catalog.

For implementation details, see the [scikit-learn clustering guide](https://scikit-learn.org/stable/modules/clustering.html) and [RapidFuzz scorer documentation](https://rapidfuzz.github.io/RapidFuzz/Usage/fuzz.html); these are technical documentation, not research articles.